# ABS SA4 Boundary Acquisition

This notebook explores and acquires the ABS Statistical Area Level 4 (SA4)
digital boundary data for ASGS Edition 4.

The raw source archive will be stored in the Bronze layer together with
provenance metadata.

In [1]:
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Working directory:", Path.cwd())

Python: D:\mycode\nsw-ev-data-pipeline\.conda\python.exe
Working directory: D:\mycode\nsw-ev-data-pipeline\notebooks


In [2]:
current_dir = Path.cwd().resolve()

for candidate in (current_dir, *current_dir.parents):
    if (candidate / "README.md").is_file() and (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Project root could not be found.")

ABS_BRONZE_DIR = PROJECT_ROOT / "data" / "bronze" / "abs"

print("Project root:", PROJECT_ROOT)
print("ABS Bronze directory:", ABS_BRONZE_DIR)

Project root: D:\mycode\nsw-ev-data-pipeline
ABS Bronze directory: D:\mycode\nsw-ev-data-pipeline\data\bronze\abs


## Step 1 — Define the ABS SA4 source

The ABS publishes ASGS Edition 4 digital boundaries in GeoPackage and
ESRI Shapefile formats.

For this project, the 2026 Statistical Area Level 4 (SA4) Shapefile
in GDA2020 will be used.

In [3]:
ABS_LANDING_PAGE = (
    "https://www.abs.gov.au/statistics/standards/"
    "australian-statistical-geography-standard-asgs/"
    "edition-4-july-2026-june-2031/"
    "access-and-downloads/digital-boundary-files"
)

ABS_RESOURCE_URL = (
    ABS_LANDING_PAGE
    + "/SA4_2026_AUST_SHP_GDA2020.zip"
)

print("Landing page:")
print(ABS_LANDING_PAGE)

print("\nResource URL:")
print(ABS_RESOURCE_URL)

Landing page:
https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files

Resource URL:
https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files/SA4_2026_AUST_SHP_GDA2020.zip


## Step 2 — Inspect the ABS download response

Before downloading the full archive, inspect the HTTP response to verify
that the resource is accessible and determine the server-provided metadata.

In [4]:
import requests

response = requests.get(
    ABS_RESOURCE_URL,
    stream=True,
    timeout=60
)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("Content-Length:", response.headers.get("Content-Length"))

response.close()

Status code: 200
Content-Type: application/zip
Content-Length: None


## Step 3 — Download the ABS archive

Download the official SA4 Shapefile archive into the Bronze layer.

The archive is streamed in chunks rather than loaded entirely into memory.
The original ZIP file is preserved unchanged as raw source data.

In [5]:
from urllib.parse import urlparse

ABS_BRONZE_DIR.mkdir(parents=True, exist_ok=True)

filename = Path(urlparse(ABS_RESOURCE_URL).path).name
archive_path = ABS_BRONZE_DIR / filename

print("Filename:", filename)
print("Output path:", archive_path)

Filename: SA4_2026_AUST_SHP_GDA2020.zip
Output path: D:\mycode\nsw-ev-data-pipeline\data\bronze\abs\SA4_2026_AUST_SHP_GDA2020.zip


In [6]:
with requests.get(
    ABS_RESOURCE_URL,
    stream=True,
    timeout=60
) as response:

    response.raise_for_status()

    content_type = response.headers.get("Content-Type")

    with archive_path.open("wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Downloaded:", archive_path)
print("Exists:", archive_path.exists())
print("Size (bytes):", archive_path.stat().st_size)
print("Content-Type:", content_type)

Downloaded: D:\mycode\nsw-ev-data-pipeline\data\bronze\abs\SA4_2026_AUST_SHP_GDA2020.zip
Exists: True
Size (bytes): 29543412
Content-Type: application/zip


## Step 4 — Validate the downloaded archive

Verify that the downloaded file is a valid ZIP archive before extraction.

In [7]:
import zipfile

is_valid_zip = zipfile.is_zipfile(archive_path)

print("Valid ZIP:", is_valid_zip)

Valid ZIP: True


## Step 5 — Inspect archive contents

Inspect the files contained in the ZIP archive before extraction.

A Shapefile dataset is composed of multiple related files, such as
`.shp`, `.shx`, `.dbf`, and `.prj`.

In [8]:
with zipfile.ZipFile(archive_path, "r") as zip_ref:
    archive_files = zip_ref.namelist()

for file_name in archive_files:
    print(file_name)

SA4_2026_AUST_GDA2020.dbf
SA4_2026_AUST_GDA2020.prj
SA4_2026_AUST_GDA2020.shp
SA4_2026_AUST_GDA2020.shx
SA4_2026_AUST_GDA2020 - Shapefile ISO Metadata.xml
SA4_2026_AUST_GDA2020.cpg


In [9]:
required_extensions = {".shp", ".shx", ".dbf", ".prj"}

found_extensions = {
    Path(file_name).suffix.lower()
    for file_name in archive_files
}

print("Found extensions:", found_extensions)
print("Required files present:",
      required_extensions.issubset(found_extensions))

Found extensions: {'.prj', '.xml', '.cpg', '.dbf', '.shx', '.shp'}
Required files present: True


## Step 6 — Extract and validate the Shapefile

Extract the ABS SA4 archive into a dedicated Bronze directory.

After extraction, verify that the core Shapefile components
(`.shp`, `.shx`, `.dbf`, and `.prj`) are present.

In [10]:
extract_dir = ABS_BRONZE_DIR / "SA4_2026_AUST_GDA2020"

extract_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Extraction directory:", extract_dir)

Extraction directory: D:\mycode\nsw-ev-data-pipeline\data\bronze\abs\SA4_2026_AUST_GDA2020


In [11]:
with zipfile.ZipFile(archive_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Extraction complete.")

Extraction complete.


In [12]:
extracted_files = list(extract_dir.iterdir())

for file_path in extracted_files:
    print(file_path.name)

SA4_2026_AUST_GDA2020 - Shapefile ISO Metadata.xml
SA4_2026_AUST_GDA2020.cpg
SA4_2026_AUST_GDA2020.dbf
SA4_2026_AUST_GDA2020.prj
SA4_2026_AUST_GDA2020.shp
SA4_2026_AUST_GDA2020.shx


In [13]:
required_files = [
    extract_dir / "SA4_2026_AUST_GDA2020.shp",
    extract_dir / "SA4_2026_AUST_GDA2020.shx",
    extract_dir / "SA4_2026_AUST_GDA2020.dbf",
    extract_dir / "SA4_2026_AUST_GDA2020.prj",
]

for file_path in required_files:
    print(
        file_path.name,
        "->",
        file_path.exists()
    )

SA4_2026_AUST_GDA2020.shp -> True
SA4_2026_AUST_GDA2020.shx -> True
SA4_2026_AUST_GDA2020.dbf -> True
SA4_2026_AUST_GDA2020.prj -> True


In [14]:
all_required_present = all(
    file_path.exists()
    for file_path in required_files
)

print(
    "All required Shapefile components present:",
    all_required_present
)

All required Shapefile components present: True


## Step 7 — Calculate SHA-256 and build provenance metadata

Calculate a SHA-256 checksum for the original ABS ZIP archive and record
provenance information describing where the data came from, when it was
retrieved, and how it was stored locally.

In [15]:
import hashlib

sha256_hash = hashlib.sha256()

with archive_path.open("rb") as f:
    for byte_block in iter(lambda: f.read(4096), b""):
        sha256_hash.update(byte_block)

archive_sha256 = sha256_hash.hexdigest()

print("SHA-256:", archive_sha256)

SHA-256: d2df15ee57dac089457fd6ff27e342d4f6286ab0dbc587fff20cf05664657323


In [16]:
from datetime import datetime, timezone

retrieved_at_utc = datetime.now(timezone.utc).isoformat()

print("Retrieved at:", retrieved_at_utc)

Retrieved at: 2026-09-14T13:10:03.340887+00:00


In [17]:
provenance = {
    "dataset_title": "Statistical Areas Level 4 (SA4) - 2026",
    "publisher": "Australian Bureau of Statistics",
    "edition": "ASGS Edition 4",
    "geography_level": "SA4",
    "reference_year": 2026,
    "coordinate_system": "GDA2020",
    "landing_page": ABS_LANDING_PAGE,
    "source_url": ABS_RESOURCE_URL,
    "retrieved_at_utc": retrieved_at_utc,
    "local_filename": archive_path.name,
    "file_size_bytes": archive_path.stat().st_size,
    "sha256": archive_sha256,
    "http_content_type": content_type,
    "archive_valid": is_valid_zip,
    "required_shapefile_components_present": all_required_present,
}

provenance

{'dataset_title': 'Statistical Areas Level 4 (SA4) - 2026',
 'publisher': 'Australian Bureau of Statistics',
 'edition': 'ASGS Edition 4',
 'geography_level': 'SA4',
 'reference_year': 2026,
 'coordinate_system': 'GDA2020',
 'landing_page': 'https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files',
 'source_url': 'https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files/SA4_2026_AUST_SHP_GDA2020.zip',
 'retrieved_at_utc': '2026-09-14T13:10:03.340887+00:00',
 'local_filename': 'SA4_2026_AUST_SHP_GDA2020.zip',
 'file_size_bytes': 29543412,
 'sha256': 'd2df15ee57dac089457fd6ff27e342d4f6286ab0dbc587fff20cf05664657323',
 'http_content_type': 'application/zip',
 'archive_valid': True,
 'required_shapefile_components_present': True}

In [18]:
import json

metadata_path = archive_path.with_suffix(".metadata.json")

with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(
        provenance,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Metadata path:", metadata_path)
print("Exists:", metadata_path.exists())

Metadata path: D:\mycode\nsw-ev-data-pipeline\data\bronze\abs\SA4_2026_AUST_SHP_GDA2020.metadata.json
Exists: True


In [19]:
with metadata_path.open("r", encoding="utf-8") as f:
    saved_metadata = json.load(f)

saved_metadata

{'dataset_title': 'Statistical Areas Level 4 (SA4) - 2026',
 'publisher': 'Australian Bureau of Statistics',
 'edition': 'ASGS Edition 4',
 'geography_level': 'SA4',
 'reference_year': 2026,
 'coordinate_system': 'GDA2020',
 'landing_page': 'https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files',
 'source_url': 'https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files/SA4_2026_AUST_SHP_GDA2020.zip',
 'retrieved_at_utc': '2026-09-14T13:10:03.340887+00:00',
 'local_filename': 'SA4_2026_AUST_SHP_GDA2020.zip',
 'file_size_bytes': 29543412,
 'sha256': 'd2df15ee57dac089457fd6ff27e342d4f6286ab0dbc587fff20cf05664657323',
 'http_content_type': 'application/zip',
 'archive_valid': True,
 'required_shapefile_components_present': True}

## Step 8 — Configure the project import path

Add the project's `src` directory to Python's module search path so that
the reusable acquisition module can be imported from this notebook.

This is required because the notebook is located in the `notebooks`
directory, while the reusable Python modules are stored under `src`.

In [24]:
import sys

src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("src path:", src_path)
print("src in sys.path:", str(src_path) in sys.path)

src path: D:\mycode\nsw-ev-data-pipeline\src
src in sys.path: True


## Step 9 — Test the reusable ABS acquisition module

Import and reload the `acquisition.abs` module, then test the reusable
ABS download function created from the notebook exploration workflow.

In [26]:
from importlib import reload
import acquisition.abs as abs_acq

reload(abs_acq)

print("Module loaded from:", abs_acq.__file__)

Module loaded from: D:\mycode\nsw-ev-data-pipeline\src\acquisition\abs.py


### Test 1 — Download the ABS archive

Test the reusable `download_abs_resource()` function.

The function should download the official ABS SA4 ZIP archive into the
Bronze directory and return both the local archive path and the HTTP
Content-Type.

In [28]:
archive_path, content_type = abs_acq.download_abs_resource(
    ABS_BRONZE_DIR
)

print("Archive:", archive_path)
print("Exists:", archive_path.exists())
print("Size:", archive_path.stat().st_size)
print("Content-Type:", content_type)

Archive: D:\mycode\nsw-ev-data-pipeline\data\bronze\abs\SA4_2026_AUST_SHP_GDA2020.zip
Exists: True
Size: 29543412
Content-Type: application/zip


### Test 2 — Validate the downloaded ABS archive

Verify that the downloaded file is a valid ZIP archive before attempting
to extract its contents.

In [29]:
reload(abs_acq)

is_valid = abs_acq.validate_zip(archive_path)

print("Valid ZIP:", is_valid)

Valid ZIP: True


### Test 3 — Extract the ABS archive

Test the reusable archive extraction function by extracting the downloaded
ABS ZIP file into a dedicated Bronze directory.

In [30]:
reload(abs_acq)

extract_dir = ABS_BRONZE_DIR / "SA4_2026_AUST_GDA2020"

extracted_dir = abs_acq.extract_abs_archive(
    archive_path,
    extract_dir
)

print("Extraction directory:", extracted_dir)
print("Exists:", extracted_dir.exists())

Extraction directory: D:\mycode\nsw-ev-data-pipeline\data\bronze\abs\SA4_2026_AUST_GDA2020
Exists: True


In [31]:
for file_path in extracted_dir.iterdir():
    print(file_path.name)

SA4_2026_AUST_GDA2020 - Shapefile ISO Metadata.xml
SA4_2026_AUST_GDA2020.cpg
SA4_2026_AUST_GDA2020.dbf
SA4_2026_AUST_GDA2020.prj
SA4_2026_AUST_GDA2020.shp
SA4_2026_AUST_GDA2020.shx


### Test 4 — Validate Shapefile components

Verify that the extracted ABS dataset contains the core Shapefile
components required for spatial processing: `.shp`, `.shx`, `.dbf`,
and `.prj`.

In [32]:
reload(abs_acq)

components_valid = abs_acq.validate_shapefile_components(
    extracted_dir
)

print(
    "Required Shapefile components present:",
    components_valid
)

Required Shapefile components present: True


In [33]:
print("Extraction directory:", extracted_dir)

Extraction directory: D:\mycode\nsw-ev-data-pipeline\data\bronze\abs\SA4_2026_AUST_GDA2020


### Test 5 — Calculate the archive SHA-256 checksum

Calculate a SHA-256 checksum for the original ABS ZIP archive.

The checksum provides a reproducible fingerprint of the raw Bronze file
and can be used to detect changes or corruption in the downloaded source.

In [34]:
reload(abs_acq)

archive_hash = abs_acq.calculate_sha256(
    archive_path
)

print("SHA-256:", archive_hash)

SHA-256: d2df15ee57dac089457fd6ff27e342d4f6286ab0dbc587fff20cf05664657323


### Test 6 — Build and save provenance metadata

Build provenance metadata for the downloaded ABS archive and save it as
a JSON file alongside the raw Bronze data.

The metadata records the source, retrieval time, file checksum, HTTP
Content-Type, and validation results.

In [35]:
reload(abs_acq)

provenance = abs_acq.build_provenance(
    archive_path,
    archive_hash,
    content_type,
    is_valid,
    components_valid,
)

provenance

{'dataset_title': 'Statistical Areas Level 4 (SA4) - 2026',
 'publisher': 'Australian Bureau of Statistics',
 'edition': 'ASGS Edition 4',
 'geography_level': 'SA4',
 'reference_year': 2026,
 'coordinate_system': 'GDA2020',
 'landing_page': 'https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files',
 'source_url': 'https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files/SA4_2026_AUST_SHP_GDA2020.zip',
 'retrieved_at_utc': '2026-09-14T13:34:07.036975+00:00',
 'local_filename': 'SA4_2026_AUST_SHP_GDA2020.zip',
 'file_size_bytes': 29543412,
 'sha256': 'd2df15ee57dac089457fd6ff27e342d4f6286ab0dbc587fff20cf05664657323',
 'http_content_type': 'application/zip',
 'archive_valid': True,
 'required_shapefile_components_present': True}

In [36]:
metadata_path = abs_acq.save_provenance(
    provenance,
    archive_path
)

print("Metadata path:", metadata_path)
print("Exists:", metadata_path.exists())

Metadata path: D:\mycode\nsw-ev-data-pipeline\data\bronze\abs\SA4_2026_AUST_SHP_GDA2020.metadata.json
Exists: True


### Test 7 — Run the complete ABS acquisition pipeline

Test the end-to-end ABS acquisition workflow.

The pipeline should:

- download the official ABS SA4 archive;
- validate the ZIP file;
- extract the archive;
- validate the required Shapefile components;
- calculate the SHA-256 checksum;
- build provenance metadata;
- save the provenance JSON file.

A successful run should return the archive path, extraction directory,
and metadata path.

In [39]:
reload(abs_acq)

result = abs_acq.acquire_abs_data(
    ABS_BRONZE_DIR
)

result

{'archive_path': WindowsPath('D:/mycode/nsw-ev-data-pipeline/data/bronze/abs/SA4_2026_AUST_SHP_GDA2020.zip'),
 'extract_dir': WindowsPath('D:/mycode/nsw-ev-data-pipeline/data/bronze/abs/SA4_2026_AUST_GDA2020'),
 'metadata_path': WindowsPath('D:/mycode/nsw-ev-data-pipeline/data/bronze/abs/SA4_2026_AUST_SHP_GDA2020.metadata.json'),
 'archive_valid': True,
 'shapefile_components_valid': True}

In [41]:
print("Archive:", result["archive_path"])
print("Archive exists:", result["archive_path"].exists())

print("Extraction directory:", result["extract_dir"])
print("Extraction directory exists:", result["extract_dir"].exists())

print("Metadata:", result["metadata_path"])
print("Metadata exists:", result["metadata_path"].exists())

print("Archive valid:", result["archive_valid"])
print("Shapefile components valid:", result["shapefile_components_valid"])

Archive: D:\mycode\nsw-ev-data-pipeline\data\bronze\abs\SA4_2026_AUST_SHP_GDA2020.zip
Archive exists: True
Extraction directory: D:\mycode\nsw-ev-data-pipeline\data\bronze\abs\SA4_2026_AUST_GDA2020
Extraction directory exists: True
Metadata: D:\mycode\nsw-ev-data-pipeline\data\bronze\abs\SA4_2026_AUST_SHP_GDA2020.metadata.json
Metadata exists: True
Archive valid: True
Shapefile components valid: True
